# 01 - Generate Synthetic Data (`The Meeting Tax`)

This notebook generates two raw CSVs:
- `data/raw/meetings_raw.csv` (attendee-level rows)
- `data/raw/pulse_raw.csv` (weekly employee pulse rows)

Design goals implemented on purpose:
1. Seniority-conditioned meeting load (with Marketing multiplier).
2. Meeting-type mix + duration/frequency patterns.
3. Productivity drops and burnout rises only after meeting-hours threshold.
4. Realistic data messiness (missing values, duplicates, outlier durations).


In [ ]:
from __future__ import annotations

from collections import OrderedDict
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd
from faker import Faker

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
fake = Faker("en_IN")
Faker.seed(RANDOM_SEED)

# Core sizing parameters
N_EMPLOYEES = 650
N_WEEKS = 130  # 2.5 years
MEETINGS_PER_WEEK_TARGET = 433

# Export paths
RAW_DATA_DIR = Path("data/raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
MEETINGS_OUT = RAW_DATA_DIR / "meetings_raw.csv"
PULSE_OUT = RAW_DATA_DIR / "pulse_raw.csv"

# Categorical mixes
DEPARTMENTS = ["Marketing", "Engineering", "Sales", "Finance", "HR", "Operations"]
DEPARTMENT_WEIGHTS = OrderedDict(
    {
        "Engineering": 0.30,
        "Sales": 0.20,
        "Marketing": 0.15,
        "Operations": 0.15,
        "Finance": 0.12,
        "HR": 0.08,
    }
)

MEETING_TYPES = ["status_update", "decision_making", "one_on_one", "brainstorm", "all_hands"]
MEETING_TYPE_WEIGHTS = [0.35, 0.20, 0.20, 0.15, 0.10]

SENIORITY_LEVELS = ["junior", "mid", "senior", "manager"]
SENIORITY_WEIGHTS = [0.55, 0.30, 0.12, 0.03]  # junior/mid-heavy org

WEEKDAY_ORDER = [0, 1, 2, 3, 4]  # Mon..Fri
WEEKDAY_WEIGHTS = [0.19, 0.22, 0.20, 0.20, 0.19]

START_WEEK = pd.Timestamp("2024-01-01")
WEEKS = pd.date_range(START_WEEK, periods=N_WEEKS, freq="W-MON")


## Employees reference table

- Faker generates person names and department labels.
- NumPy controls the seniority distribution and salary noise.
- Salary bands match the spec and are clipped to realistic ranges.


In [ ]:
def generate_salary(seniority: str, size: int) -> np.ndarray:
    if seniority == "junior":
        values = rng.normal(600_000, 80_000, size)
        return np.clip(values, 400_000, 800_000)
    if seniority == "mid":
        values = rng.normal(1_150_000, 150_000, size)
        return np.clip(values, 800_000, 1_500_000)
    if seniority == "senior":
        values = rng.normal(2_150_000, 250_000, size)
        return np.clip(values, 1_500_000, 2_800_000)
    values = rng.normal(3_500_000, 400_000, size)
    return np.clip(values, 2_500_000, 4_500_000)


employees = pd.DataFrame(
    {
        "employee_id": np.arange(1, N_EMPLOYEES + 1),
        "employee_name": [fake.name() for _ in range(N_EMPLOYEES)],
        "department": [fake.random_element(elements=DEPARTMENT_WEIGHTS) for _ in range(N_EMPLOYEES)],
        "seniority": rng.choice(SENIORITY_LEVELS, size=N_EMPLOYEES, p=SENIORITY_WEIGHTS),
    }
)

employees["annual_salary"] = 0.0
for level in SENIORITY_LEVELS:
    mask = employees["seniority"] == level
    employees.loc[mask, "annual_salary"] = generate_salary(level, int(mask.sum()))

employees["annual_salary"] = employees["annual_salary"].round(2)
employees["hourly_rate"] = (employees["annual_salary"] / 1920.0).round(2)

dept_to_employee_ids = employees.groupby("department")["employee_id"].apply(list).to_dict()
all_employee_ids = employees["employee_id"].to_numpy()

employees.head()


## Meetings + attendee-level generation

Why this shape:
- Meeting types are sampled with exact target shares.
- Duration and attendee-count generators follow type-specific formulas.
- Recurrence flags are generated per type (weekly-heavy for status/1:1, monthly for all-hands).
- Dates are weighted by weekday to intentionally front-load Tue-Thu.
- A small share of recurring meetings is force-adjusted to `attendee_count > 8` so downstream expensive-recurring patterns are discoverable.


In [ ]:
meeting_type_config = {
    "status_update": {
        "duration_mean": 45,
        "duration_std": 15,
        "recurring_prob": 0.60,
        "attendee_gen": lambda n=1: np.clip(rng.normal(5, 2, n), 2, 15).round().astype(int),
    },
    "decision_making": {
        "duration_mean": 50,
        "duration_std": 15,
        "recurring_prob": 0.35,
        "attendee_gen": lambda n=1: np.clip(rng.normal(6, 2, n), 2, 15).round().astype(int),
    },
    "one_on_one": {
        "duration_mean": 30,
        "duration_std": 10,
        "recurring_prob": 0.70,
        "attendee_gen": lambda n=1: np.full(n, 2, dtype=int),
    },
    "brainstorm": {
        "duration_mean": 60,
        "duration_std": 20,
        "recurring_prob": 0.30,
        "attendee_gen": lambda n=1: np.clip(rng.normal(5, 1.5, n), 2, 12).round().astype(int),
    },
    "all_hands": {
        "duration_mean": 45,
        "duration_std": 10,
        "recurring_prob": 0.90,
        "attendee_gen": lambda n=1: np.clip(rng.normal(60, 15, n), 20, 120).round().astype(int),
    },
}

recurring_pools = {m_type: [f"SER_{m_type}_{fake.unique.lexify(text='??????')}" for _ in range(250)] for m_type in MEETING_TYPES}

meeting_records = []
meeting_id = 1

for week_start in WEEKS:
    weekly_meeting_count = int(np.clip(rng.normal(MEETINGS_PER_WEEK_TARGET, 35), 320, 560))

    for _ in range(weekly_meeting_count):
        meeting_type = str(rng.choice(MEETING_TYPES, p=MEETING_TYPE_WEIGHTS))
        department = str(rng.choice(list(DEPARTMENT_WEIGHTS.keys()), p=list(DEPARTMENT_WEIGHTS.values())))
        organizer_seniority = str(rng.choice(SENIORITY_LEVELS, p=[0.20, 0.40, 0.25, 0.15]))

        config = meeting_type_config[meeting_type]
        duration = int(np.clip(rng.normal(config["duration_mean"], config["duration_std"]), 15, 180))
        attendee_count = int(config["attendee_gen"](1)[0])

        is_recurring = bool(rng.random() < config["recurring_prob"])
        recurring_series_id = None
        if is_recurring:
            recurring_series_id = str(rng.choice(recurring_pools[meeting_type]))

        weighted_weekday = int(rng.choice(WEEKDAY_ORDER, p=WEEKDAY_WEIGHTS))
        day_start = (week_start + timedelta(days=weighted_weekday)).date()
        meeting_date = fake.date_between_dates(date_start=day_start, date_end=day_start)

        meeting_records.append(
            {
                "meeting_id": meeting_id,
                "date": pd.Timestamp(meeting_date),
                "department": department,
                "meeting_type": meeting_type,
                "duration_mins": duration,
                "attendee_count": attendee_count,
                "organizer_seniority": organizer_seniority,
                "is_recurring": is_recurring,
                "recurring_series_id": recurring_series_id,
            }
        )
        meeting_id += 1

meetings = pd.DataFrame(meeting_records)

# Intentional outlier meetings: 0.5% very long workshops
n_outliers = max(1, int(len(meetings) * 0.005))
outlier_idx = rng.choice(meetings.index.to_numpy(), size=n_outliers, replace=False)
meetings.loc[outlier_idx, "duration_mins"] = rng.integers(240, 481, size=n_outliers)

# Force 4% of recurring meetings to be >8 attendees (detectable recurring-overload pattern)
recurring_idx = meetings.index[meetings["is_recurring"]].to_numpy()
target_large_recurring = int(len(recurring_idx) * 0.04)
if target_large_recurring > 0:
    eligible = meetings.index[
        meetings["is_recurring"]
        & meetings["meeting_type"].isin(["status_update", "decision_making", "brainstorm", "all_hands"])
        & (meetings["attendee_count"] <= 8)
    ].to_numpy()
    adjust_n = min(target_large_recurring, len(eligible))
    if adjust_n > 0:
        adjust_idx = rng.choice(eligible, size=adjust_n, replace=False)
        meetings.loc[adjust_idx, "attendee_count"] = rng.integers(9, 14, size=adjust_n)

# Intentional missing duration values: 3% MCAR, avoiding outlier rows
n_missing_duration = max(1, int(len(meetings) * 0.03))
remaining_for_missing = meetings.index.difference(outlier_idx).to_numpy()
missing_duration_idx = rng.choice(remaining_for_missing, size=n_missing_duration, replace=False)
meetings.loc[missing_duration_idx, "duration_mins"] = np.nan

meetings["is_disproportionately_expensive"] = (
    meetings["meeting_type"].isin(["status_update", "one_on_one"]) & (meetings["attendee_count"] > 8)
)

# Build attendee-level rows (clean, before duplicate injection)
attendee_pairs = []
for row in meetings.itertuples(index=False):
    attendee_n = int(row.attendee_count)
    if row.meeting_type == "all_hands":
        chosen_ids = rng.choice(all_employee_ids, size=min(attendee_n, len(all_employee_ids)), replace=False)
    else:
        same_dept_ids = np.array(dept_to_employee_ids.get(row.department, []), dtype=int)
        if len(same_dept_ids) == 0:
            same_dept_ids = all_employee_ids

        same_n = int(round(attendee_n * 0.70))
        same_n = max(1, min(same_n, len(same_dept_ids), attendee_n))
        cross_n = attendee_n - same_n

        same_pick = rng.choice(same_dept_ids, size=same_n, replace=False)
        if cross_n > 0:
            cross_pool = np.setdiff1d(all_employee_ids, same_pick, assume_unique=False)
            cross_pick = rng.choice(cross_pool, size=min(cross_n, len(cross_pool)), replace=False)
            chosen_ids = np.concatenate([same_pick, cross_pick])
        else:
            chosen_ids = same_pick

    for emp_id in chosen_ids:
        attendee_pairs.append((row.meeting_id, int(emp_id)))

meeting_attendees = pd.DataFrame(attendee_pairs, columns=["meeting_id", "employee_id"])

meetings_raw_clean = (
    meeting_attendees
    .merge(meetings, on="meeting_id", how="left")
    .merge(
        employees[["employee_id", "department", "seniority", "hourly_rate"]].rename(
            columns={"department": "attendee_department", "seniority": "attendee_seniority"}
        ),
        on="employee_id",
        how="left",
    )
)

meetings_raw_clean["row_cost"] = (
    (meetings_raw_clean["duration_mins"] / 60.0) * meetings_raw_clean["hourly_rate"]
).round(2)

# Intentional duplicate rows: 1% of attendee-level data
n_dupes = max(1, int(len(meetings_raw_clean) * 0.01))
duplicate_rows = meetings_raw_clean.sample(n=n_dupes, random_state=RANDOM_SEED, replace=False)
meetings_raw = pd.concat([meetings_raw_clean, duplicate_rows], ignore_index=True)

meetings_raw["date"] = pd.to_datetime(meetings_raw["date"]).dt.date
meetings_raw = meetings_raw[
    [
        "meeting_id",
        "date",
        "department",
        "meeting_type",
        "duration_mins",
        "attendee_count",
        "organizer_seniority",
        "is_recurring",
        "recurring_series_id",
        "is_disproportionately_expensive",
        "employee_id",
        "attendee_seniority",
        "attendee_department",
        "hourly_rate",
        "row_cost",
    ]
]

meetings_raw.head()


## Weekly pulse generation

Why this logic:
- Weekly meeting-hours are generated directly from seniority-conditioned distributions.
- Marketing gets a 1.3x multiplier to encode the "meeting-heavy" department effect.
- Productivity and burnout equations only activate above the 15-hour threshold to reflect nonlinear overload effects.
- Missingness is injected intentionally (3% productivity, 2% burnout).


In [ ]:
n_pulse = N_EMPLOYEES * N_WEEKS

pulse = pd.DataFrame(
    {
        "employee_id": np.repeat(employees["employee_id"].to_numpy(), N_WEEKS),
        "week": np.tile(WEEKS.date, N_EMPLOYEES),
    }
)

pulse = pulse.merge(
    employees[["employee_id", "department", "seniority"]],
    on="employee_id",
    how="left",
)

hours = np.zeros(n_pulse)

def sample_hours(mask: pd.Series, mean: float, std: float, low: float, high: float) -> None:
    sampled = rng.normal(mean, std, int(mask.sum()))
    hours[mask.to_numpy()] = np.clip(sampled, low, high)

sample_hours(pulse["seniority"] == "junior", 5.5, 2.0, 0, 12)
sample_hours(pulse["seniority"] == "mid", 8.0, 2.5, 0, 15)
sample_hours(pulse["seniority"] == "senior", 13.0, 3.0, 4, 20)
sample_hours(pulse["seniority"] == "manager", 18.0, 3.5, 8, 28)

marketing_mask = pulse["department"] == "Marketing"
hours[marketing_mask.to_numpy()] *= 1.3

pulse["hours_in_meetings"] = np.round(hours, 2)

prod_noise = rng.normal(0, 0.5, n_pulse)
burn_noise = rng.normal(0, 0.5, n_pulse)

prod = 7 - np.maximum(0, pulse["hours_in_meetings"].to_numpy() - 15) * 0.15 + prod_noise
burn = 3 + np.maximum(0, pulse["hours_in_meetings"].to_numpy() - 15) * 0.2 + burn_noise

pulse["self_reported_productivity"] = np.rint(np.clip(prod, 1, 10)).astype(float)
pulse["self_reported_burnout"] = np.rint(np.clip(burn, 1, 10)).astype(float)

# Intentional missingness in pulse data
n_missing_prod = max(1, int(len(pulse) * 0.03))
n_missing_burn = max(1, int(len(pulse) * 0.02))

prod_missing_idx = rng.choice(pulse.index.to_numpy(), size=n_missing_prod, replace=False)
burn_missing_idx = rng.choice(pulse.index.to_numpy(), size=n_missing_burn, replace=False)

pulse.loc[prod_missing_idx, "self_reported_productivity"] = np.nan
pulse.loc[burn_missing_idx, "self_reported_burnout"] = np.nan

pulse = pulse[
    [
        "employee_id",
        "week",
        "department",
        "seniority",
        "hours_in_meetings",
        "self_reported_productivity",
        "self_reported_burnout",
    ]
]

pulse.head()


## Export + sanity checks + annual cost benchmark

The annual cost check is intentionally explicit:
- Cost is computed from attendee-level `row_cost`.
- Duplicate logging errors are excluded from the benchmark check to avoid contamination.
- INR cost is converted to USD (assumption printed) for comparison with Flowtrace/Reclaim's ~$25K-$30K benchmark.
- If the estimate is very far from benchmark, a clear warning is printed.


In [ ]:
meetings_raw.to_csv(MEETINGS_OUT, index=False)
pulse.to_csv(PULSE_OUT, index=False)

print("=== FILE EXPORTS ===")
print(f"meetings_raw -> {MEETINGS_OUT.resolve()}")
print(f"pulse_raw    -> {PULSE_OUT.resolve()}")

print("\n=== ROW COUNTS ===")
print(f"employees                : {len(employees):,}")
print(f"meetings (distinct)      : {len(meetings):,}")
print(f"meeting_attendees (clean): {len(meetings_raw_clean):,}")
print(f"meetings_raw (with dupes): {len(meetings_raw):,}")
print(f"pulse_raw                : {len(pulse):,}")

print("\n=== MISSINGNESS CHECK ===")
print(f"duration_mins missing %          : {meetings_raw['duration_mins'].isna().mean() * 100:.2f}%")
print(f"self_reported_productivity miss %: {pulse['self_reported_productivity'].isna().mean() * 100:.2f}%")
print(f"self_reported_burnout miss %     : {pulse['self_reported_burnout'].isna().mean() * 100:.2f}%")

dupe_rate = meetings_raw.duplicated().mean() * 100
print(f"attendee-level exact duplicate % : {dupe_rate:.2f}%")

print("\n=== PATTERN SANITY CHECKS ===")
print("Meeting type share (%):")
print((meetings['meeting_type'].value_counts(normalize=True) * 100).round(2))

weekday_labels = pd.to_datetime(meetings['date']).dt.day_name()
print("\nMeeting weekday share (%):")
print((weekday_labels.value_counts(normalize=True) * 100).round(2).reindex(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']))

recurring_over8_share = (
    meetings.loc[meetings['is_recurring'], 'attendee_count'].gt(8).mean() * 100
)
print(f"\nRecurring meetings with attendee_count > 8: {recurring_over8_share:.2f}%")

heavy_hours = pulse['hours_in_meetings'].to_numpy()
p90 = np.percentile(heavy_hours, 90)
p98 = np.percentile(heavy_hours, 98)
print(f"hours_in_meetings p90: {p90:.2f} | p98: {p98:.2f}")
if not (8.5 <= p90 <= 11.5 and 17.0 <= p98 <= 22.0):
    print("WARNING: Meeting-hour percentile shape drifted from target; nudge seniority means if needed.")

print("\nProductivity/Burnout by meeting-hour bucket:")
pulse_summary = pulse.copy()
pulse_summary['hour_bucket'] = pd.cut(
    pulse_summary['hours_in_meetings'],
    bins=[-0.1, 5, 10, 15, 20, 30],
    labels=['0-5', '5-10', '10-15', '15-20', '20+']
)
print(
    pulse_summary.groupby('hour_bucket', observed=True)[['self_reported_productivity', 'self_reported_burnout']]
    .mean()
    .round(2)
)

# Annual meeting cost benchmark check (exclude injected duplicate rows)
years_covered = N_WEEKS / 52.0
annual_cost_per_employee_inr = meetings_raw_clean['row_cost'].sum(skipna=True) / (N_EMPLOYEES * years_covered)

INR_PER_USD = 83.0
annual_cost_per_employee_usd = annual_cost_per_employee_inr / INR_PER_USD

benchmark_low, benchmark_high = 25_000, 30_000

print("\n=== ANNUAL MEETING COST SANITY CHECK ===")
print(f"Assumed FX: 1 USD = {INR_PER_USD:.1f} INR")
print(f"Projected annual cost per employee: INR {annual_cost_per_employee_inr:,.0f} | USD ${annual_cost_per_employee_usd:,.0f}")
print(f"Published benchmark (Flowtrace/Reclaim): USD ${benchmark_low:,.0f}-${benchmark_high:,.0f}")

if annual_cost_per_employee_usd < 15_000 or annual_cost_per_employee_usd > 45_000:
    print("WILDLY OFF BENCHMARK: adjust salary/duration/frequency parameters before proceeding.")
elif benchmark_low <= annual_cost_per_employee_usd <= benchmark_high:
    print("Within benchmark range.")
else:
    print("Near benchmark but outside target band: review assumptions before downstream analysis.")
